# M3 · 대시보드 v1 확인
② 사고재해자수, ③ 사고사망자수, ④ 사업장수, ⑤ 사망만인율을 같은 업종×규모 칸에서 본다. 차트는 개인 파일로 저장한다.

In [ ]:
from pathlib import Path
import pandas as pd
import plotly.express as px
from IPython.display import display
ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/'pyproject.toml').exists()),None)
assert ROOT is not None
PERSONAL=ROOT/'data'/'personal'
df=pd.read_parquet(PERSONAL/'processed'/'dashboard_metrics.parquet')
FIGURES=PERSONAL/'figures';FIGURES.mkdir(parents=True,exist_ok=True)
assert len(df)==300
print('집계 칸',len(df),'사망만인율 결측',df.fatality_rate.isna().sum())

## 업종과 규모를 골라 보기

In [ ]:
display(df[['industry_major','industry_middle','size_code','accident_injured',
            'accident_deaths','establishments','fatality_rate']].head(12))
major=sorted(df.industry_major.unique())[0]
subset=df[df.industry_major==major]
display(subset.groupby('size_code',as_index=False)[['accident_injured','accident_deaths']].sum())
fig=px.bar(subset,x='size_code',y='accident_injured',color='industry_middle',
           title=f'{major} · 규모별 사고재해자수 (2025 집계)',barmode='group')
fig.write_html(FIGURES/'dashboard_v1_injured.html',include_plotlyjs='directory')
display(fig)

## 분모와 결측 확인

In [ ]:
check=df.groupby('size_code',as_index=False).agg(
    establishments=('establishments','sum'),injured=('accident_injured','sum'),
    deaths=('accident_deaths','sum'),rate_missing=('fatality_rate',lambda s:int(s.isna().sum())))
display(check)
fig=px.scatter(df.dropna(subset=['fatality_rate']),x='establishments',y='fatality_rate',
               color='industry_major',hover_data=['industry_middle','size_code'],
               title='사업장 수와 공식 사망만인율 · 업종×규모 칸')
fig.write_html(FIGURES/'dashboard_v1_rate.html',include_plotlyjs='directory')
display(fig)

## 대시보드 설계 판단

- ②~⑤의 업종×규모 300칸을 공통 표로 두고 사고재해자수, 사고사망자수, 사업장수, 사망만인율을 각각 원래 단위대로 보여주는 시안을 만들었다. 사망만인율은 업종·규모 한 칸을 선택했을 때만 표시하고 여러 칸을 평균·합산하지 않는다.
- ⑤의 결측 14칸은 결측으로 남겼다. 그래프에서 빠지는 경우 선택 범위와 결측 수를 따로 알 수 있게 해야 비교 대상이 달라지는 일을 줄일 수 있다.
- 사고재해자·사고사망자는 건수, 사업장수는 개소, 사망만인율은 임금근로자 1만 명당 비율이다. 사업장수는 사망만인율의 계산 분모가 아니다.
- 선택 예시 업종은 정렬 순서상 건설업이다. 이 정적 노트북 차트는 탐색 화면이고 Streamlit 상호작용·예외 처리는 M8에서 따로 확인한다.
- SIF 사례 빈도도 별도 시각화로 표시하되 2025년 집계와 합산하지 않는다. 사례 수는 해당 아카이브에 실린 건수이지 전체 업종 위험률이 아니다.

### 화면을 볼 때 주의할 점

- 기준일·단위·출처를 그래프 바로 옆에 붙인다. 빈칸을 0으로 보이게 하지 않는다.
- 기준기간과 SIF 아카이브 버전이 달라 기간 비교를 주장하지 않는다.
- 실제 사용 화면의 입력, 필터, 챗봇 근거 표시는 M8 자동 화면검사 및 수동 질의기록과 구분해서 설명한다.
